# Competitive Product Perception Benchmark

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Analyze multi-dimensional** using Snowflake Cortex AI
2. **Build production pipelines** with SQL and SENTIMENT()
3. **Create data structures** for Multi-dimensional analysis
4. **Implement business logic** for communications intelligence
5. **Generar insights** with window functions and aggregations
6. **Build interactive dashboards** for stakeholder reporting

---

## What You'll Build

A **competitive product perception benchmark** that provides:
- Multi-dimensional competitor comparison across efficacy, safety, cost, convenience
- Automated data collection and processing
- Real-time analysis and insights
- Interactive visualization dashboard
- Production-ready SQL for scale

---

## Technical Requirements

- **Snowflake account** with Cortex AI enabled
- **Approximately 12-15 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features

- `AI_SENTIMENT()` - Primary AI function
- Window functions - Time-series analysis
- Aggregations - Summary statistics
- CTEs - Complex query logic
- `CASE` statements - Classification

Let's begin!

---

## Paso 1: Configuración del Entorno

### Qué Vamos a Crear

- **Database**: `PERCEPTION_BENCHMARK_DB`
- **Schema**: `PUBLIC`
- **Warehouse**: `COMPUTE_WH`

### Business Challenge

Multi-dimensional competitor comparison across efficacy, safety, cost, convenience

### Why This Matters

Modern communications require data-driven insights:
- **Speed**: Real-time analysis vs manual review
- **Scale**: Process thousands of items automatically
- **Consistency**: Same rules applied every time
- **Intelligence**: AI-powered insights

In [ ]:
-- Create environment
CREATE DATABASE IF NOT EXISTS COMPETITIVE_PERCEPTION_DB;
CREATE SCHEMA IF NOT EXISTS COMPETITIVE_PERCEPTION_DB.PUBLIC;
USE SCHEMA COMPETITIVE_PERCEPTION_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT 
    CURRENT_DATABASE() as database_name,
    CURRENT_SCHEMA() as schema_name,
    CURRENT_WAREHOUSE() as warehouse_name,
    'Competitive Product Perception Benchmark Environment Ready!' as status;

---

## Paso 2: Define Data Structure

### Tables

1. **`product_reviews`** - Primary data source
2. **`perception_scores`** - Analysis results
3. **`competitive_benchmarks`** - Aggregated insights

### Data Flow

1. **Ingest** → Collect data from sources
2. **Process** → Apply AI functions
3. **Analyze** → Calculate metrics
4. **Visualize** → Present insights

In [ ]:
-- Primary data table
CREATE OR REPLACE TABLE product_reviews (
    record_id VARCHAR(50) PRIMARY KEY,
    source VARCHAR(100),
    content TEXT,
    created_date DATE,
    metadata VARIANT,
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
) COMMENT = 'Synthetic review content with product + dimension context';

-- Sentiment + AI scores
CREATE OR REPLACE TABLE perception_scores (
    analysis_id VARCHAR(50) PRIMARY KEY,
    record_id VARCHAR(50) REFERENCES product_reviews(record_id),
    product_name VARCHAR(100),
    dimension VARCHAR(50),
    sentiment_score FLOAT,
    sentiment_category VARCHAR(50),
    sentiment_intensity FLOAT,
    analyzed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- LLM theme extraction
CREATE OR REPLACE TABLE perception_themes (
    insight_id VARCHAR(50) PRIMARY KEY,
    record_id VARCHAR(50) REFERENCES product_reviews(record_id),
    product_name VARCHAR(100),
    dimension VARCHAR(50),
    perception_theme VARCHAR(100),
    executive_summary TEXT,
    severity_flag VARCHAR(50),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- Aggregated insights table
CREATE OR REPLACE TABLE competitive_benchmarks (
    insight_id VARCHAR(50) PRIMARY KEY,
    date_period DATE,
    metric_name VARCHAR(100),
    metric_value FLOAT,
    trend VARCHAR(20),
    calculated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

SELECT 'Tables created successfully!' as status;

---

## Paso 3: Generar Datos Sintéticos Data

### Qué Vamos a Crear

- **1,000 records** for demonstration
- **Realistic content** matching use case
- **Last 30 days** of data
- **Multiple sources** and categories

### Synthetic Data Approach

Using Snowflake's `GENERATOR()` and `UNIFORM()` functions to create realistic test data.

In [ ]:
-- Generar datos sintéticos competitor perception data
INSERT INTO product_reviews
WITH products AS (
    SELECT * FROM (VALUES
        ('Xarelto', 'Bayer'),
        ('Eylea', 'Bayer'),
        ('Stivarga', 'Bayer'),
        ('Mounjaro', 'Eli Lilly'),
        ('Trulicity', 'Eli Lilly'),
        ('Adempas', 'Bayer'),
        ('Jardiance', 'Boehringer Ingelheim'),
        ('Farxiga', 'AstraZeneca')
    ) t(product, manufacturer)
),
perception_dimensions AS (
    SELECT * FROM (VALUES
        ('efficacy'), ('safety'), ('cost'), ('convenience'), ('access'), ('experience')
    ) t(dimension)
),
content_templates AS (
    SELECT * FROM (VALUES
        ('Outstanding clinical outcomes for TYPE product'),
        ('Safety monitoring indicates fewer side effects than peers'),
        ('Pricing pressure emerging across payer negotiations'),
        ('Patients praise simple dosing schedules'),
        ('Market buzz highlights strong efficacy story'),
        ('Coverage friction reported by payers'),
        ('HCPs cite confidence in safety margin'),
        ('Patients mention onboarding hurdles')
    ) t(template)
)
SELECT 
    'REC_' || LPAD(SEQ4()::VARCHAR, 8, '0') as record_id,
    p.product as source,
    REPLACE(ct.template, 'TYPE', p.product) as content,
    DATEADD('day', -FLOOR(UNIFORM(1, 30, RANDOM())), CURRENT_DATE()) as created_date,
    OBJECT_CONSTRUCT(
        'product', p.product,
        'manufacturer', p.manufacturer,
        'dimension', d.dimension,
        'channel', ARRAY_CONSTRUCT('clinical', 'patient', 'provider', 'payer')[FLOOR(UNIFORM(0, 4, RANDOM()))],
        'region', ARRAY_CONSTRUCT('US', 'EU', 'LATAM', 'APAC')[FLOOR(UNIFORM(0, 4, RANDOM()))]
    ) as metadata,
    CURRENT_TIMESTAMP() as ingested_at
FROM TABLE(GENERATOR(ROWCOUNT => 2000)) g
CROSS JOIN products p
CROSS JOIN perception_dimensions d
CROSS JOIN content_templates ct
WHERE UNIFORM(0, 100, RANDOM()) < 6
QUALIFY ROW_NUMBER() OVER (ORDER BY UNIFORM(0, 1, RANDOM())) <= 1200;

SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT source) as sources,
    MIN(created_date) as earliest_date,
    MAX(created_date) as latest_date
FROM product_reviews;

---

## Paso 4: Score Perception with `AI_SENTIMENT()`

### Qué Vamos a Hacer
We apply Cortex sentiment scoring to every review to quantify how positively or negatively each product is discussed across dimensions.

### Why This Matters
- **Objective baselines** for communications teams instead of manual tagging
- **Comparability** across efficacy, safety, cost, convenience, etc.
- **Speed** — thousands of reviews processed in seconds

### Technical Details
`AI_SENTIMENT()` returns a **-1 to +1 score**. We categorize it into strategic bands and persist results in `perception_scores` for downstream analytics.

In [ ]:
-- Apply Cortex AI sentiment analysis to product reviews
-- Verified via Snowflake docs 2025-01: AI_SENTIMENT returns OBJECT
CREATE OR REPLACE TABLE perception_scores AS
WITH scored_reviews AS (
    SELECT 
        r.record_id,
        r.metadata:product::STRING as product_name,
        r.metadata:dimension::STRING as dimension,
        r.content,
        AI_SENTIMENT(r.content)['categories'][0]['sentiment']::STRING as sentiment_label,
        ROW_NUMBER() OVER (ORDER BY r.record_id) as row_id
    FROM product_reviews r
    QUALIFY ROW_NUMBER() OVER (ORDER BY r.record_id) <= 900
)
SELECT 
    'ANL_' || LPAD(row_id::VARCHAR, 8, '0') as analysis_id,
    record_id,
    product_name,
    dimension,
    -- Convert sentiment to numeric score (Use Case 28 pattern)
    CASE sentiment_label
        WHEN 'positive' THEN 1.0
        WHEN 'neutral' THEN 0.0
        WHEN 'negative' THEN -1.0
        WHEN 'mixed' THEN -0.5
        ELSE -1.0
    END as sentiment_score,
    CASE sentiment_label
        WHEN 'positive' THEN 'Highly Positive'
        WHEN 'neutral' THEN 'Neutral'
        WHEN 'negative' THEN 'Negative'
        WHEN 'mixed' THEN 'Negative'
        ELSE 'Highly Negative'
    END as sentiment_category,
    ABS(CASE sentiment_label
        WHEN 'positive' THEN 1.0
        WHEN 'neutral' THEN 0.0
        WHEN 'negative' THEN -1.0
        WHEN 'mixed' THEN -0.5
        ELSE -1.0
    END) as sentiment_intensity,
    CURRENT_TIMESTAMP() as analyzed_at
FROM scored_reviews;

SELECT 
    sentiment_category,
    COUNT(*) as review_count,
    ROUND(AVG(sentiment_score), 3) as avg_sentiment,
    ROUND(AVG(sentiment_intensity), 3) as avg_intensity
FROM perception_scores
GROUP BY sentiment_category
ORDER BY avg_sentiment DESC;

---

## Paso 5: Extract Themes with `AI_COMPLETE()` + `SUMMARIZE()`

### Qué Vamos a Hacer
We enrich the most negative reviews with LLM-generated classifications and executive summaries to understand why perception drops.

### Why This Matters
- Converts raw sentiment into **actionable themes** (safety concern, cost pressure, etc.)
- Provides **ready-to-share summaries** for executive updates
- Aligns analytics with real messaging adjustments

### Technical Details
`AI_COMPLETE()` classifies each review into one threat label, while `AI_SUMMARIZE()` condenses the text into a single recommendation.

In [ ]:
-- LLM theme extraction for negative reviews
CREATE OR REPLACE TABLE perception_themes AS
WITH negative_reviews AS (
    SELECT 
        ps.analysis_id,
        ps.record_id,
        ps.product_name,
        ps.dimension,
        ps.sentiment_score,
        pr.content,
        ROW_NUMBER() OVER (ORDER BY ps.sentiment_score) as row_id
    FROM perception_scores ps
    JOIN product_reviews pr ON ps.record_id = pr.record_id
    WHERE ps.sentiment_score < 0
)
SELECT 
    'THEME_' || LPAD(row_id::VARCHAR, 8, '0') as insight_id,
    record_id,
    product_name,
    dimension,
    AI_COMPLETE(
        'mistral-large2',
        'You are a global communications analyst. Classify this competitor perception into ONE label (Efficacy Risk, Safety Concern, Cost Pressure, Convenience Friction, Market Buzz). Only return the label. Review: ' || content
    ) as perception_theme,
    SNOWFLAKE.CORTEX.SUMMARIZE(content) as executive_summary,
    CASE 
        WHEN sentiment_score <= -0.5 THEN 'High Risk'
        WHEN sentiment_score <= -0.2 THEN 'Medium Risk'
        ELSE 'Monitor'
    END as severity_flag,
    CURRENT_TIMESTAMP() as created_at
FROM negative_reviews
QUALIFY row_id <= 200;

SELECT 
    perception_theme,
    severity_flag,
    COUNT(*) as issue_count
FROM perception_themes
GROUP BY perception_theme, severity_flag
ORDER BY issue_count DESC;

---

## Paso 6: Multi-Dimensional Perception Analysis

### Qué Vamos a Hacer
Analyze product performance across efficacy, safety, cost, convenience, access, and experience using the Cortex-enhanced scores.

### Why This Matters
- **Holistic view** of competitive strengths and weaknesses
- **Dimension-level benchmarks** for each product
- **Prioritized actions** for communications and field teams

### Technical Details
We aggregate `perception_scores` by product + dimension, calculate variability, and assign traffic-light ratings based on sentiment.

In [ ]:
-- Multi-Dimensional Perception Analysis by Product
WITH dimension_scores AS (
    SELECT 
        p.product_name,
        p.dimension,
        COUNT(*) as review_count,
        ROUND(AVG(p.sentiment_score), 3) as avg_perception,
        ROUND(STDDEV(p.sentiment_score), 3) as perception_stddev,
        ROUND(MIN(p.sentiment_score), 3) as worst_perception,
        ROUND(MAX(p.sentiment_score), 3) as best_perception,
        ROUND(AVG(p.sentiment_intensity), 3) as avg_intensity
    FROM perception_scores p
    GROUP BY p.product_name, p.dimension
)
SELECT 
    product_name,
    dimension,
    review_count,
    avg_perception,
    perception_stddev,
    worst_perception,
    best_perception,
    avg_intensity,
    RANK() OVER (PARTITION BY dimension ORDER BY avg_perception DESC) as dimension_rank,
    CASE 
        WHEN avg_perception >= 0.5 THEN 'Strong'
        WHEN avg_perception >= 0.2 THEN 'Moderate'
        WHEN avg_perception >= -0.2 THEN 'Weak'
        ELSE 'Critical'
    END as perception_level
FROM dimension_scores
ORDER BY dimension, dimension_rank;

---

## Paso 7: Competitive Positioning Matrix

### Qué Vamos a Hacer

Create a **competitive positioning matrix** showing where each product stands relative to competitors across key dimensions.

### Why This Matters

- **Visual positioning**: See at a glance where products compete
- **Gap identification**: Find white space opportunities
- **Strategic planning**: Inform product positioning and messaging

### Matrix Dimensions

Comparing products on two critical axes:
- **X-axis**: Efficacy perception
- **Y-axis**: Safety perception
- **Size**: Review volume (market visibility)

In [ ]:
-- Competitive Positioning Matrix
WITH product_dimensions AS (
    SELECT 
        p.product_name,
        p.dimension,
        ROUND(AVG(p.sentiment_score), 3) as avg_score
    FROM perception_scores p
    WHERE p.dimension IN ('efficacy', 'safety', 'cost', 'convenience')
    GROUP BY p.product_name, p.dimension
),
pivoted AS (
    SELECT 
        product_name,
        MAX(CASE WHEN dimension = 'efficacy' THEN avg_score END) as efficacy_score,
        MAX(CASE WHEN dimension = 'safety' THEN avg_score END) as safety_score,
        MAX(CASE WHEN dimension = 'cost' THEN avg_score END) as cost_score,
        MAX(CASE WHEN dimension = 'convenience' THEN avg_score END) as convenience_score
    FROM product_dimensions
    GROUP BY product_name
)
SELECT 
    product_name,
    efficacy_score,
    safety_score,
    cost_score,
    convenience_score,
    ROUND((efficacy_score + safety_score + cost_score + convenience_score) / 4, 3) as overall_score,
    CASE 
        WHEN efficacy_score >= 0 AND safety_score >= 0 THEN 'Leader (High Efficacy + High Safety)'
        WHEN efficacy_score >= 0 AND safety_score < 0 THEN 'Powerful (High Efficacy, Safety Concerns)'
        WHEN efficacy_score < 0 AND safety_score >= 0 THEN 'Safe (Lower Efficacy, High Safety)'
        ELSE 'Challenged (Low Efficacy + Low Safety)'
    END as positioning_quadrant,
    CASE 
        WHEN cost_score >= 0 AND convenience_score >= 0 THEN 'High Value'
        WHEN cost_score >= 0 AND convenience_score < 0 THEN 'Affordable'
        WHEN cost_score < 0 AND convenience_score >= 0 THEN 'Convenient'
        ELSE 'Price Sensitive'
    END as value_positioning,
    RANK() OVER (ORDER BY overall_score DESC) as market_rank
FROM pivoted
ORDER BY overall_score DESC;

---

## Paso 8: Gap Analysis - Where Competitors Excel

### Qué Vamos a Hacer

Identify **specific areas where competitor products are perceived more favorably** than Bayer products.

### Why This Matters

- **Threat identification**: Know where competitors have advantages
- **Defensive strategy**: Prioritize areas needing improvement or messaging
- **Realistic assessment**: Understand competitive landscape honestly

### Analysis Focus

- Which dimensions do competitors win on?
- How large are the perception gaps?
- Which competitor products pose the biggest threats?

In [ ]:
-- Gap Analysis: Where Competitors Excel
WITH dimension_leaders AS (
    SELECT 
        p.dimension,
        p.product_name,
        ROUND(AVG(p.sentiment_score), 3) as avg_perception,
        COUNT(*) as review_count,
        RANK() OVER (PARTITION BY p.dimension ORDER BY AVG(p.sentiment_score) DESC) as rank_in_dimension
    FROM perception_scores p
    WHERE p.dimension IS NOT NULL
    GROUP BY p.dimension, p.product_name
),
novo_performance AS (
    SELECT 
        dimension,
        MAX(avg_perception) as novo_best_score
    FROM dimension_leaders
    WHERE product_name IN ('Xarelto', 'Eylea', 'Stivarga')
    GROUP BY dimension
),
competitor_advantages AS (
    SELECT 
        dl.dimension,
        dl.product_name as competitor_product,
        dl.avg_perception as competitor_score,
        dl.review_count,
        np.novo_best_score,
        ROUND(dl.avg_perception - np.novo_best_score, 3) as perception_gap,
        dl.rank_in_dimension as market_rank
    FROM dimension_leaders dl
    INNER JOIN novo_performance np ON dl.dimension = np.dimension
    WHERE dl.product_name NOT IN ('Xarelto', 'Eylea', 'Stivarga')
    AND dl.avg_perception > np.novo_best_score
)
SELECT 
    dimension,
    competitor_product,
    competitor_score,
    novo_best_score,
    perception_gap,
    market_rank,
    CASE 
        WHEN perception_gap >= 0.5 THEN 'Critical Gap'
        WHEN perception_gap >= 0.3 THEN 'Significant Gap'
        WHEN perception_gap >= 0.1 THEN 'Moderate Gap'
        ELSE 'Minor Gap'
    END as gap_severity
FROM competitor_advantages
ORDER BY perception_gap DESC, dimension
LIMIT 15;

---

## Paso 9: Strength Analysis - Where Novo Products Excel

### Qué Vamos a Hacer

Identify **areas where Bayer products are perceived more favorably** than competitors.

### Why This Matters

- **Leverage strengths**: Emphasize these in marketing and communications
- **Confidence building**: Know where you have competitive advantages
- **Offensive strategy**: Use strengths to gain market share

### Analysis Focus

- Which dimensions do Novo products lead on?
- How significant are the advantages?
- Which Novo products have the strongest positioning?

In [ ]:
-- Strength Analysis: Where Novo Products Excel
WITH dimension_scores AS (
    SELECT 
        p.product_name,
        p.dimension,
        ROUND(AVG(p.sentiment_score), 3) as avg_perception,
        COUNT(*) as review_count
    FROM perception_scores p
    WHERE p.dimension IS NOT NULL
    GROUP BY p.product_name, p.dimension
),
dimension_benchmarks AS (
    SELECT 
        dimension,
        ROUND(AVG(avg_perception), 3) as market_average,
        ROUND(MAX(avg_perception), 3) as market_best
    FROM dimension_scores
    WHERE product_name NOT IN ('Xarelto', 'Eylea', 'Stivarga')
    GROUP BY dimension
),
novo_advantages AS (
    SELECT 
        ds.product_name,
        ds.dimension,
        ds.avg_perception as novo_score,
        ds.review_count,
        db.market_average as competitor_avg,
        db.market_best as competitor_best,
        ROUND(ds.avg_perception - db.market_average, 3) as advantage_vs_average,
        ROUND(ds.avg_perception - db.market_best, 3) as advantage_vs_best
    FROM dimension_scores ds
    INNER JOIN dimension_benchmarks db ON ds.dimension = db.dimension
    WHERE ds.product_name IN ('Xarelto', 'Eylea', 'Stivarga')
    AND ds.avg_perception > db.market_average
)
SELECT 
    product_name,
    dimension,
    novo_score,
    competitor_avg,
    competitor_best,
    advantage_vs_average,
    advantage_vs_best,
    review_count,
    CASE 
        WHEN advantage_vs_best > 0 THEN 'Market Leader'
        WHEN advantage_vs_average >= 0.3 THEN 'Strong Advantage'
        WHEN advantage_vs_average >= 0.1 THEN 'Moderate Advantage'
        ELSE 'Slight Advantage'
    END as strength_level,
    RANK() OVER (PARTITION BY product_name ORDER BY advantage_vs_average DESC) as strength_rank
FROM novo_advantages
ORDER BY advantage_vs_average DESC
LIMIT 15;

---

## Paso 10: Competitive Rankings & Strategic Insights

### Qué Vamos a Hacer

Create **overall competitive rankings** across all dimensions and generate strategic insights.

### Why This Matters

- **Executive summary**: Quick view of competitive landscape
- **Priority setting**: Which products and dimensions need attention
- **Success tracking**: Benchmark for measuring improvements

### Key Metrics

- Overall competitive scores by product
- Dimension-specific rankings
- Market share of positive perception
- Strategic recommendations

In [ ]:
-- Manufacturer-Level Perception Comparison
WITH manufacturer_scores AS (
    SELECT 
        p.product_name,
        CASE 
            WHEN p.product_name IN ('Xarelto', 'Eylea', 'Stivarga') THEN 'Bayer'
            WHEN p.product_name IN ('Mounjaro', 'Trulicity') THEN 'Eli Lilly'
            WHEN p.product_name IN ('Adempas', 'Nubeqa') THEN 'Bayer'
            ELSE 'Other Manufacturer'
        END as manufacturer,
        ROUND(AVG(p.sentiment_score), 3) as avg_sentiment,
        COUNT(*) as review_count,
        COUNT(DISTINCT p.dimension) as dimension_count
    FROM perception_scores p
    GROUP BY p.product_name, manufacturer
)
SELECT 
    manufacturer,
    COUNT(DISTINCT product_name) as product_count,
    SUM(review_count) as total_reviews,
    ROUND(AVG(avg_sentiment), 3) as manufacturer_avg_sentiment,
    ROUND(MIN(avg_sentiment), 3) as weakest_product_score,
    ROUND(MAX(avg_sentiment), 3) as strongest_product_score,
    ROUND(STDDEV(avg_sentiment), 3) as product_consistency,
    SUM(dimension_count) as total_dimensions_covered,
    CASE 
        WHEN AVG(avg_sentiment) >= 0.3 THEN 'Market Leader'
        WHEN AVG(avg_sentiment) >= 0 THEN 'Competitive'
        WHEN AVG(avg_sentiment) >= -0.3 THEN 'Challenged'
        ELSE 'Struggling'
    END as market_position
FROM manufacturer_scores
GROUP BY manufacturer
ORDER BY manufacturer_avg_sentiment DESC;

In [ ]:
-- Calculate daily metrics
INSERT INTO competitive_benchmarks
SELECT 
    'INS_' || LPAD(ROW_NUMBER() OVER (ORDER BY created_date), 8, '0') as insight_id,
    created_date as date_period,
    'daily_average' as metric_name,
    ROUND(AVG(p.sentiment_score), 3) as metric_value,
    CASE 
        WHEN AVG(p.sentiment_score) > LAG(AVG(p.sentiment_score)) OVER (ORDER BY created_date) THEN 'up'
        WHEN AVG(p.sentiment_score) < LAG(AVG(p.sentiment_score)) OVER (ORDER BY created_date) THEN 'down'
        ELSE 'stable'
    END as trend,
    CURRENT_TIMESTAMP() as calculated_at
FROM product_reviews r
INNER JOIN perception_scores p ON r.record_id = p.record_id
GROUP BY created_date;

SELECT * FROM competitive_benchmarks ORDER BY date_period DESC LIMIT 10;

In [ ]:
# Competitive Product Perception Dashboard
import streamlit as st
import pandas as pd
import altair as alt
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title(" Competitive Product Perception Benchmark")
st.markdown("### Multi-dimensional competitive analysis across efficacy, safety, cost, and convenience")

# Dependency validation
try:
    dependency_check = session.sql("""
        SELECT 
            (SELECT COUNT(*) FROM product_reviews) as review_count,
            (SELECT COUNT(*) FROM perception_scores) as score_count,
            (SELECT COUNT(*) FROM perception_themes) as theme_count
    """).collect()[0]
    if dependency_check['REVIEW_COUNT'] == 0 or dependency_check['SCORE_COUNT'] == 0:
        st.error(" Missing data! Run Cells 2-10 before launching the dashboard.")
        st.stop()
except Exception as e:
    st.error(" Required tables not found. Run Cells 2-10 first.")
    st.info(f"Error details: {str(e)[:200]}")
    st.stop()

# Create tabs
tab1, tab2, tab3, tab4 = st.tabs([" Overview", " Positioning Matrix", " Gaps & Strengths", " Details"])

with tab1:
    st.subheader("Key Metrics")
    col1, col2, col3, col4 = st.columns(4)
    total_reviews = dependency_check['REVIEW_COUNT']
    total_products = session.sql("SELECT COUNT(DISTINCT source) as cnt FROM product_reviews").collect()[0]['CNT']
    avg_perception = session.sql("SELECT ROUND(AVG(sentiment_score), 3) as avg FROM perception_scores").collect()[0]['AVG'] or 0.0
    themes_generated = dependency_check['THEME_COUNT']
    with col1:
        st.metric("Total Reviews", f"{total_reviews:,}", help="Product reviews analyzed")
    with col2:
        st.metric("Products Tracked", total_products, help="Unique products in analysis")
    with col3:
        perception_emoji = "" if avg_perception >= 0.3 else "" if avg_perception >= -0.3 else ""
        st.metric("Avg Sentiment", f"{avg_perception:.3f}", delta=perception_emoji)
    with col4:
        st.metric("LLM Themes", themes_generated, help="Negative-review insights extracted")
    st.markdown("---")
    st.subheader(" Perception Distribution by Product")
    try:
        product_perception = session.sql("""
            SELECT 
                r.source as product,
                COUNT(*) as review_count,
                ROUND(AVG(p.sentiment_score), 3) as avg_sentiment,
                ROUND(AVG(p.sentiment_intensity), 3) as avg_intensity
            FROM product_reviews r
            INNER JOIN perception_scores p ON r.record_id = p.record_id
            GROUP BY r.source
            ORDER BY avg_sentiment DESC
            LIMIT 15
        """).to_pandas()
        if not product_perception.empty:
            chart = alt.Chart(product_perception).mark_bar().encode(
                x=alt.X('AVG_SENTIMENT:Q', title='Average Sentiment', scale=alt.Scale(domain=[-1, 1])),
                y=alt.Y('PRODUCT:N', title='Product', sort='-x'),
                color=alt.condition(alt.datum.AVG_SENTIMENT > 0, alt.value('#00CC96'), alt.value('#EF553B')),
                tooltip=['PRODUCT:N', 'AVG_SENTIMENT:Q', 'REVIEW_COUNT:Q', 'AVG_INTENSITY:Q']
            ).properties(width=700, height=400, title='Product Sentiment Scores')
            zero_line = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='gray', strokeDash=[2, 2]).encode(x='x:Q')
            st.altair_chart(chart + zero_line, use_container_width=True)
            col1, col2 = st.columns(2)
            with col1:
                st.markdown("** Top Perceived Products:**")
                for _, row in product_perception.head(3).iterrows():
                    st.success(f"**{row['PRODUCT']}**: {row['AVG_SENTIMENT']:.3f} ({row['REVIEW_COUNT']} reviews)")
            with col2:
                st.markdown("** Lowest Perceived Products:**")
                for _, row in product_perception.tail(3).iterrows():
                    st.warning(f"**{row['PRODUCT']}**: {row['AVG_SENTIMENT']:.3f} ({row['REVIEW_COUNT']} reviews)")
        else:
            st.info("No product perception data available. Run Cells 6-8.")
    except Exception as e:
        st.warning(f"Product perception data not available: {str(e)}")

with tab2:
    st.subheader(" Competitive Positioning Matrix")
    st.markdown("Products positioned by **efficacy** vs **safety** perception")
    try:
        positioning = session.sql("""
            WITH product_dimensions AS (
                SELECT 
                    p.product_name,
                    p.dimension,
                    ROUND(AVG(p.sentiment_score), 3) as avg_score
                FROM perception_scores p
                WHERE p.dimension IN ('efficacy', 'safety')
                GROUP BY p.product_name, p.dimension
            )
            SELECT 
                product_name,
                MAX(CASE WHEN dimension = 'efficacy' THEN avg_score END) as efficacy_score,
                MAX(CASE WHEN dimension = 'safety' THEN avg_score END) as safety_score
            FROM product_dimensions
            GROUP BY product_name
            HAVING efficacy_score IS NOT NULL AND safety_score IS NOT NULL
        """).to_pandas()
        if not positioning.empty:
            scatter = alt.Chart(positioning).mark_circle(size=200).encode(
                x=alt.X('EFFICACY_SCORE:Q', title='Efficacy Sentiment', scale=alt.Scale(domain=[-1, 1])),
                y=alt.Y('SAFETY_SCORE:Q', title='Safety Sentiment', scale=alt.Scale(domain=[-1, 1])),
                color=alt.condition((alt.datum.EFFICACY_SCORE > 0) & (alt.datum.SAFETY_SCORE > 0), alt.value('#00CC96'), alt.value('#EF553B')),
                tooltip=['PRODUCT_NAME:N', 'EFFICACY_SCORE:Q', 'SAFETY_SCORE:Q']
            ).properties(width=700, height=500, title='Competitive Positioning Matrix')
            h_line = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(color='gray', strokeDash=[5, 5]).encode(y='y:Q')
            v_line = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='gray', strokeDash=[5, 5]).encode(x='x:Q')
            text = scatter.mark_text(align='left', dx=10, dy=-10, fontSize=10).encode(text='PRODUCT_NAME:N')
            st.altair_chart(scatter + h_line + v_line + text, use_container_width=True)
        else:
            st.info("No positioning data available. Ensure metadata includes efficacy/safety dimensions.")
    except Exception as e:
        st.warning(f"Positioning matrix not available: {str(e)}")

with tab3:
    st.subheader(" Competitive Gaps & Strengths")
    col1, col2 = st.columns(2)
    with col1:
        st.markdown("###  Where Competitors Excel")
        try:
            gaps = session.sql("""
                SELECT 
                    r.source as product,
                    ROUND(AVG(p.sentiment_score), 3) as avg_sentiment,
                    COUNT(*) as review_count
                FROM product_reviews r
                INNER JOIN perception_scores p ON r.record_id = p.record_id
                WHERE r.source NOT IN ('Xarelto', 'Eylea', 'Stivarga')
                GROUP BY r.source
                HAVING avg_sentiment > (
                    SELECT AVG(p2.sentiment_score)
                    FROM product_reviews r2
                    INNER JOIN perception_scores p2 ON r2.record_id = p2.record_id
                    WHERE r2.source IN ('Xarelto', 'Eylea', 'Stivarga')
                )
                ORDER BY avg_sentiment DESC
                LIMIT 5
            """).to_pandas()
            if not gaps.empty:
                for _, row in gaps.iterrows():
                    st.warning(f"**{row['PRODUCT']}** — Sentiment: {row['AVG_SENTIMENT']:.3f} ({row['REVIEW_COUNT']} reviews)")
            else:
                st.success(" No significant competitive gaps detected!")
        except Exception as e:
            st.info(f"Gap analysis not available: {str(e)}")
    with col2:
        st.markdown("###  Where Novo Products Excel")
        try:
            strengths = session.sql("""
                SELECT 
                    r.source as product,
                    ROUND(AVG(p.sentiment_score), 3) as avg_sentiment,
                    COUNT(*) as review_count
                FROM product_reviews r
                INNER JOIN perception_scores p ON r.record_id = p.record_id
                WHERE r.source IN ('Xarelto', 'Eylea', 'Stivarga')
                GROUP BY r.source
                ORDER BY avg_sentiment DESC
            """).to_pandas()
            if not strengths.empty:
                for _, row in strengths.iterrows():
                    emoji = '' if row['AVG_SENTIMENT'] >= 0.3 else ('' if row['AVG_SENTIMENT'] >= 0 else '')
                    st.info(f"**{row['PRODUCT']}** {emoji} — Sentiment: {row['AVG_SENTIMENT']:.3f} ({row['REVIEW_COUNT']} reviews)")
            else:
                st.info("No Novo product data available")
        except Exception as e:
            st.info(f"Strength analysis not available: {str(e)}")

with tab4:
    st.subheader(" Detailed Product Reviews")
    col1, col2 = st.columns(2)
    perception_filter = col1.selectbox("Filter by Sentiment", ["All", "Positive (≥ 0.3)", "Neutral (-0.3 to 0.3)", "Negative (≤ -0.3)"])
    limit = col2.slider("Number of records", 10, 100, 50)
    if perception_filter == "Positive (≥ 0.3)":
        where_clause = "WHERE p.sentiment_score >= 0.3"
    elif perception_filter == "Negative (≤ -0.3)":
        where_clause = "WHERE p.sentiment_score <= -0.3"
    elif perception_filter == "Neutral (-0.3 to 0.3)":
        where_clause = "WHERE p.sentiment_score BETWEEN -0.3 AND 0.3"
    else:
        where_clause = ""
    try:
        detailed_data = session.sql(f"""
            SELECT 
                r.source as product,
                SUBSTRING(r.content, 1, 120) as review_preview,
                p.dimension,
                p.sentiment_category,
                ROUND(p.sentiment_score, 3) as sentiment_score,
                ROUND(p.sentiment_intensity, 3) as sentiment_intensity,
                r.created_date
            FROM product_reviews r
            INNER JOIN perception_scores p ON r.record_id = p.record_id
            {where_clause}
            ORDER BY r.created_date DESC
            LIMIT {limit}
        """).to_pandas()
        if not detailed_data.empty:
            st.dataframe(detailed_data, use_container_width=True, hide_index=True)
            st.download_button(
                label=" Download Detailed Data",
                data=detailed_data.to_csv(index=False),
                file_name="competitive_perception_details.csv",
                mime="text/csv"
            )
        else:
            st.info("No records match the selected filters")
    except Exception as e:
        st.error(f"Error loading detailed data: {str(e)}")

st.markdown("---")
st.success(" Competitive Perception Dashboard operational | Data current")

with st.expander("ℹ About This Dashboard"):
    st.markdown("""
    ### Competitive Product Perception Benchmark
    - **Overview Tab:** Key metrics and perception distribution
    - **Positioning Matrix:** Efficacy vs Safety visualization
    - **Gaps & Strengths:** Competitive threats and Novo wins
    - **Details:** Filterable review table with export option

    **Sentiment Bands:**
    - Positive ≥ 0.3 | Neutral between -0.3 and 0.3 | Negative ≤ -0.3
    """)

---

## ✅ Tutorial Complete!

### What You've Built
- ✅ Cortex-powered sentiment scoring across six perception dimensions
- ✅ LLM theme extraction + executive summaries for negative reviews
- ✅ Competitive positioning, gap, and strength analytics
- ✅ Streamlit dashboard with built-in dependency validation

### Key Techniques
- `AI_SENTIMENT()` for scoring perception
- `AI_COMPLETE()` for zero-shot theme labels
- `SNOWFLAKE.CORTEX.SUMMARIZE()` for executive-ready narratives
- Window functions, CTEs, and aggregations for benchmarking
- Streamlit + Altair for interactive storytelling

### Next Steps for Production
1. Connect to real competitive intelligence feeds
2. Automate refresh cadence and alerting
3. Extend LLM prompts with brand-specific tone
4. Feed insights into executive dashboards and PR workflows

**Estimated runtime**: varies by data volume and warehouse size.

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto
This cell will **completely remove** all objects created in this tutorial:
- Drops the `COMPETITIVE_PERCEPTION_DB` database (and all tables/models within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use
Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions
**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data and models will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS COMPETITIVE_PERCEPTION_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;